In [1]:
import pandas as pd

data = pd.read_csv("/content/data-demodpr2025 (1).csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2002 entries, 0 to 2001
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   conversation_id_str      2002 non-null   int64  
 1   created_at               2002 non-null   object 
 2   favorite_count           2002 non-null   int64  
 3   full_text                2002 non-null   object 
 4   id_str                   2002 non-null   int64  
 5   image_url                166 non-null    object 
 6   in_reply_to_screen_name  1381 non-null   object 
 7   lang                     2002 non-null   object 
 8   location                 0 non-null      float64
 9   quote_count              2002 non-null   int64  
 10  reply_count              2002 non-null   int64  
 11  retweet_count            2002 non-null   int64  
 12  tweet_url                2002 non-null   object 
 13  user_id_str              2002 non-null   int64  
 14  username                

In [2]:
data.head(5)

,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username
0,1972592507340075208,Mon Sep 29 16:25:01 +0000 2025,0,@PartaiSocmed @Heraloebss Tanpa sebab? Sebabny...,1972699143023009923,NaN,PartaiSocmed,in,NaN,0,1,0,https://x.com/undefined/status/197269914302300...,1286070654236676096,NaN
1,1972603026893594671,Mon Sep 29 10:03:05 +0000 2025,1,Sebab MBG adalah salah satu instrument politik...,1972603026893594671,NaN,NaN,in,NaN,0,0,0,https://x.com/undefined/status/197260302689359...,1434455521633849346,NaN
2,1972538807577944275,Mon Sep 29 05:47:54 +0000 2025,0,Kl mau fair hrsnya yg menggaungkan hashtag bub...,1972538807577944275,NaN,NaN,in,NaN,0,0,0,https://x.com/undefined/status/197253880757794...,123814864,NaN
3,1972072038618878321,Mon Sep 29 03:47:46 +0000 2025,1,@Andria75777 Bubarkan DPR,1972508575684805032,NaN,Andria75777,in,NaN,0,0,0,https://x.com/undefined/status/197250857568480...,1472824248762728450,NaN
4,1972454974673355100,Mon Sep 29 00:14:47 +0000 2025,4,dia yang hadiri kumpul2 di banyuwangi dgn eks ...,1972454974673355100,NaN,NaN,in,NaN,0,2,0,https://x.com/undefined/status/197245497467335...,1710523287443574784,NaN


In [3]:
df  = pd.DataFrame(data[['full_text']])
df.head(5)

,full_text
0,@PartaiSocmed @Heraloebss Tanpa sebab? Sebabny...
1,Sebab MBG adalah salah satu instrument politik...
2,Kl mau fair hrsnya yg menggaungkan hashtag bub...
3,@Andria75777 Bubarkan DPR
4,dia yang hadiri kumpul2 di banyuwangi dgn eks ...


In [4]:
import pandas as pd
total_baris_awal = len(df)
#Bersihkan spasi tersembunyi
df['full_text'] = df['full_text'].str.strip()

#Hitung jumlah kata
df['word_count'] = df['full_text'].apply(lambda x: len(str(x).split()))

# Pisahkan data
tweet_pendek = df[df['word_count'] <= 2]
tweet_panjang = df[df['word_count'] > 2]

jumlah_duplikat = tweet_panjang.duplicated(subset=['full_text']).sum()

# Hapus duplikat HANYA di tweet panjang
tweet_panjang_clean = tweet_panjang.drop_duplicates(subset=['full_text'], keep='first')
df = pd.concat([tweet_pendek, tweet_panjang_clean]).sort_index()
df = df.drop(columns=['word_count'])

total_baris_akhir = len(df)
jumlah_dihapus = total_baris_awal - total_baris_akhir

# --- TAMPILKAN LAPORAN ---
print("=== LAPORAN PEMBERSIHAN DATA ===")
print(f"Total baris awal          : {total_baris_awal}")
print(f"Jumlah duplikasi terdeteksi : {jumlah_duplikat} (hanya pada tweet > 2 kata)")
print(f"Jumlah baris yang dihapus   : {jumlah_dihapus}")
print(f"Total baris sekarang        : {total_baris_akhir}")

=== LAPORAN PEMBERSIHAN DATA ===
Total baris awal          : 2002
Jumlah duplikasi terdeteksi : 5 (hanya pada tweet > 2 kata)
Jumlah baris yang dihapus   : 5
Total baris sekarang        : 1997


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1997 entries, 0 to 2001
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   full_text  1997 non-null   object
dtypes: object(1)
memory usage: 31.2+ KB


In [6]:
import re
import string
import nltk

# Fungsi untuk menghapus URL
def remove_URL(tweet):
    if tweet is not None and isinstance(tweet, str):
        url = re.compile(r'https?://\S+|www\.\S+')
        return url.sub(r'', tweet)
    else:
        return tweet

# Fungsi untuk menghapus HTML
def remove_html(tweet):
    if tweet is not None and isinstance(tweet, str):
        html = re.compile(r'<.*?>')
        return html.sub(r'', tweet)
    else:
        return tweet

# Fungsi untuk menghapus emoji
def remove_emoji(tweet):
    if tweet is not None and isinstance(tweet, str):
        emoji_pattern = re.compile("["
            u"\U0001F600-\U0001F64F"  # emoticons
            u"\U0001F300-\U0001F5FF"  # symbols & pictographs
            u"\U0001F680-\U0001F6FF"  # transport & map symbols
            u"\U0001F700-\U0001F77F"  # alchemical symbols
            u"\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
            u"\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
            u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
            u"\U0001FA00-\U0001FA6F"  # Chess Symbols
            u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
            u"\U0001F004-\U0001F0CF"  # Additional emoticons
            u"\U0001F1E0-\U0001F1FF"  # flags
                               "]+", flags=re.UNICODE)
        return emoji_pattern.sub(r'', tweet)
    else:
        return tweet

# Fungsi untuk menghapus simbol
def remove_symbols(tweet):
    if tweet is not None and isinstance(tweet, str):
        tweet = re.sub(r'[^a-zA-Z0-9\s]', '', tweet)
    return tweet

# Fungsi untuk menghapus angka
def remove_numbers(tweet):
    if tweet is not None and isinstance(tweet, str):
        tweet = re.sub(r'\d', '', tweet)
    return tweet

# Fungsi hapus username
def remove_usernames(text):
    return re.sub(r'@\w+', '', text)

df['cleaning'] = df['full_text'].apply(lambda x: remove_URL(x))
df['cleaning'] = df['cleaning'].apply(lambda x: remove_usernames(x))
df['cleaning'] = df['cleaning'].apply(lambda x: remove_html(x))
df['cleaning'] = df['cleaning'].apply(lambda x: remove_emoji(x))
df['cleaning'] = df['cleaning'].apply(lambda x: remove_symbols(x))
df['cleaning'] = df['cleaning'].apply(lambda x: remove_numbers(x))

df.head(5)

,full_text,cleaning
0,@PartaiSocmed @Heraloebss Tanpa sebab? Sebabny...,Tanpa sebab Sebabnya seruan bubarkan dpr aga...
1,Sebab MBG adalah salah satu instrument politik...,Sebab MBG adalah salah satu instrument politik...
2,Kl mau fair hrsnya yg menggaungkan hashtag bub...,Kl mau fair hrsnya yg menggaungkan hashtag bub...
3,@Andria75777 Bubarkan DPR,Bubarkan DPR
4,dia yang hadiri kumpul2 di banyuwangi dgn eks ...,dia yang hadiri kumpul di banyuwangi dgn eks P...


In [7]:
def case_folding(text):
    if isinstance(text, str):
        lowercase_text = text.lower()
        return lowercase_text
    else:
        return text

df['case_folding'] = df['cleaning'].apply(case_folding)
df.head(5)

,full_text,cleaning,case_folding
0,@PartaiSocmed @Heraloebss Tanpa sebab? Sebabny...,Tanpa sebab Sebabnya seruan bubarkan dpr aga...,tanpa sebab sebabnya seruan bubarkan dpr aga...
1,Sebab MBG adalah salah satu instrument politik...,Sebab MBG adalah salah satu instrument politik...,sebab mbg adalah salah satu instrument politik...
2,Kl mau fair hrsnya yg menggaungkan hashtag bub...,Kl mau fair hrsnya yg menggaungkan hashtag bub...,kl mau fair hrsnya yg menggaungkan hashtag bub...
3,@Andria75777 Bubarkan DPR,Bubarkan DPR,bubarkan dpr
4,dia yang hadiri kumpul2 di banyuwangi dgn eks ...,dia yang hadiri kumpul di banyuwangi dgn eks P...,dia yang hadiri kumpul di banyuwangi dgn eks p...


In [8]:
import pandas as pd
import requests
from io import BytesIO

# Fungsi penggantian kata tidak baku
def replace_taboo_words(text, kamus_tidak_baku):
    if isinstance(text, str):
        words = text.split()
        replaced_words = []
        kalimat_baku = []
        kata_diganti = []
        kata_tidak_baku_hash = []

        for word in words:
            if word in kamus_tidak_baku:
                baku_word = kamus_tidak_baku[word]
                if isinstance(baku_word, str) and all(char.isalpha() for char in baku_word):
                    replaced_words.append(baku_word)
                    kalimat_baku.append(baku_word)
                    kata_diganti.append(word)
                    kata_tidak_baku_hash.append(hash(word))
            else:
                replaced_words.append(word)
        replaced_text = ' '.join(replaced_words)
    else:
        replaced_text = ''
        kalimat_baku = []
        kata_diganti = []
        kata_tidak_baku_hash = []

    return replaced_text, kalimat_baku, kata_diganti, kata_tidak_baku_hash

# Baca dataset kamu (pastikan df sudah tersedia)
data = pd.DataFrame(df[['full_text','cleaning','case_folding']])
data.head()

,full_text,cleaning,case_folding
0,@PartaiSocmed @Heraloebss Tanpa sebab? Sebabny...,Tanpa sebab Sebabnya seruan bubarkan dpr aga...,tanpa sebab sebabnya seruan bubarkan dpr aga...
1,Sebab MBG adalah salah satu instrument politik...,Sebab MBG adalah salah satu instrument politik...,sebab mbg adalah salah satu instrument politik...
2,Kl mau fair hrsnya yg menggaungkan hashtag bub...,Kl mau fair hrsnya yg menggaungkan hashtag bub...,kl mau fair hrsnya yg menggaungkan hashtag bub...
3,@Andria75777 Bubarkan DPR,Bubarkan DPR,bubarkan dpr
4,dia yang hadiri kumpul2 di banyuwangi dgn eks ...,dia yang hadiri kumpul di banyuwangi dgn eks P...,dia yang hadiri kumpul di banyuwangi dgn eks p...


In [9]:
# Unduh dan baca kamus dari GitHub
url = "https://github.com/analysisdatasentiment/kamus_kata_baku/raw/main/kamuskatabaku.xlsx"
response = requests.get(url)
file_excel = BytesIO(response.content)
kamus_data = pd.read_excel(file_excel)

# Buat dictionary dari kamus
kamus_tidak_baku_dict = dict(zip(kamus_data['tidak_baku'], kamus_data['kata_baku']))

In [10]:
# Terapkan fungsi normalisasi
data[['normalisasi', 'Kata_Baku', 'Kata_Tidak_Baku', 'Kata_Tidak_Baku_Hash']] = data['case_folding'].apply(
    lambda x: pd.Series(replace_taboo_words(x, kamus_tidak_baku_dict))
)

# Ambil kolom yang relevan
df = pd.DataFrame(data[['full_text','cleaning','case_folding','normalisasi']])
df.head(5)

,full_text,cleaning,case_folding,normalisasi
0,@PartaiSocmed @Heraloebss Tanpa sebab? Sebabny...,Tanpa sebab Sebabnya seruan bubarkan dpr aga...,tanpa sebab sebabnya seruan bubarkan dpr aga...,tanpa sebab sebabnya seruan bubarkan dpr agar ...
1,Sebab MBG adalah salah satu instrument politik...,Sebab MBG adalah salah satu instrument politik...,sebab mbg adalah salah satu instrument politik...,sebab mbak adalah salah satu instrument politi...
2,Kl mau fair hrsnya yg menggaungkan hashtag bub...,Kl mau fair hrsnya yg menggaungkan hashtag bub...,kl mau fair hrsnya yg menggaungkan hashtag bub...,kalau mau fair harusnya yang menggaungkan hash...
3,@Andria75777 Bubarkan DPR,Bubarkan DPR,bubarkan dpr,bubarkan dpr
4,dia yang hadiri kumpul2 di banyuwangi dgn eks ...,dia yang hadiri kumpul di banyuwangi dgn eks P...,dia yang hadiri kumpul di banyuwangi dgn eks p...,dia yang hadiri kumpul di banyuwangi dengan ek...


In [11]:
import nltk
from nltk.corpus import stopwords

# Pastikan resource stopwords sudah terunduh
nltk.download('stopwords')

# 1. Ambil daftar stopword bahasa Indonesia
list_stopwords = set(stopwords.words('indonesian'))

# 2. Daftar Negasi Super Lengkap (Sangat Komprehensif)
negation_words = {
    # Formal
    'tidak', 'bukan', 'jangan', 'tak', 'tiada', 'tanpa', 'belum', 'kurang', 'bukanlah', 'tidaklah', 'janganlah', 'sayangnya',
    # Non-formal / Slang / Dialek
    'gak', 'nggak', 'ga', 'enggak', 'engga', 'ngga', 'kagak', 'ndak', 'ora', 'mboten', 'moh', 'ogah', 'males', 'nyatanya', 'njet', 'mohon',
    # Singkatan & Typo Media Sosial (Vowel repetitions & common typos)
    'tdk', 'bkn', 'jgn', 'gk', 'g', 'gx', 'dpt', 'bukanx', 'tdak', 'tiddak', 'ngak', 'nggaa', 'nggaklah', 'gakk', 'nggakada',
    'gaa', 'gakk', 'ndaklah', 'tdkk', 'jgann', 'jg', 'gag', 'gax', 'nggk', 'malesnya', 'blm', 'blum', 'lum', 'blumada',
    # Kata Gabungan Negasi Umum
    'gada', 'gpp', 'gapapa', 'gausah', 'gakusah', 'ndakusah', 'ndakpapa', 'nggakpapa', 'gakpapa', 'gkada', 'gaada'
}

# 3. Hapus kata negasi dari list_stopwords agar tidak ikut terhapus dari teks
filtered_stopwords = [word for word in list_stopwords if word not in negation_words]

# 4. Fungsi untuk menghapus stopwords
def remove_custom_stopwords(text):
    if isinstance(text, str):
        words = text.split()
        cleaned_words = [word for word in words if word.lower() in negation_words or word.lower() not in filtered_stopwords]
        return ' '.join(cleaned_words)
    return text

# 5. Terapkan pada kolom normalisasi
df['text_final'] = df['normalisasi'].apply(remove_custom_stopwords)

display(df[['normalisasi', 'text_final']].head(10))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,normalisasi,text_final
0,tanpa sebab sebabnya seruan bubarkan dpr agar ...,tanpa seruan bubarkan dpr maksulkan gibran tak...
1,sebab mbak adalah salah satu instrument politi...,mbak salah instrument politik genk solo sontol...
2,kalau mau fair harusnya yang menggaungkan hash...,fair menggaungkan hashtag bubarkan dpr kemarin...
3,bubarkan dpr,bubarkan dpr
4,dia yang hadiri kumpul di banyuwangi dengan ek...,hadiri kumpul banyuwangi eks pki
5,bubarkan polri adalah pilihan terbaik tak perl...,bubarkan polri pilihan terbaik tak repotrepot ...
6,gas terus bubarkan polri supaya tidak repot ga...,gas bubarkan polri tidak repot ganti polri per...
7,makanya tidak perlu demo bubarkan dpr kasih sa...,tidak demo bubarkan dpr kasih gaji umr bubar
8,analisa ferry irwandi masuk akal memang akun i...,analisa ferry irwandi masuk akal akun provokas...
9,lengserkan semua ya dan dpr bubarkan bangsa be...,lengserkan ya dpr bubarkan bangsa hancur pemil...


In [12]:
# Menyimpan dataframe hasil pembersihan ke file CSV
output_filename = 'data_preprocessing_withstopwordromoval.csv'
df.to_csv(output_filename, index=False)

print(f"Data berhasil disimpan ke {output_filename}")
# Menampilkan 5 baris pertama dari file yang disimpan untuk verifikasi
display(df[['full_text', 'text_final']].head())

Data berhasil disimpan ke data_preprocessing_withstopwordromoval.csv


,full_text,text_final
0,@PartaiSocmed @Heraloebss Tanpa sebab? Sebabny...,tanpa seruan bubarkan dpr maksulkan gibran tak...
1,Sebab MBG adalah salah satu instrument politik...,mbak salah instrument politik genk solo sontol...
2,Kl mau fair hrsnya yg menggaungkan hashtag bub...,fair menggaungkan hashtag bubarkan dpr kemarin...
3,@Andria75777 Bubarkan DPR,bubarkan dpr
4,dia yang hadiri kumpul2 di banyuwangi dgn eks ...,hadiri kumpul banyuwangi eks pki
